# Week 1 — Exercise 1: Individual Metadata Creation

**Goal:** Apply DataCite and schema.org metadata standards to a publicly available dataset.

**Dataset chosen:** `OnlyTrivial_dt` — a tabular dataset of CK software metrics for Java classes, published on Zenodo under DOI [10.5281/zenodo.6800385](https://doi.org/10.5281/zenodo.6800385).

The dataset contains CK software metrics (cbo, wmc, dit, rfc, lcom, …) computed on Java classes, with a binary `refactoring` label indicating whether a trivial class-level refactoring was applied. It is used to train ML models that predict refactoring opportunities.

This notebook:
1. Inspects the CSV to extract factual metadata (row count, column count, size, schema).
2. Builds **DataCite 4.5** metadata as XML.
3. Builds **schema.org** metadata as JSON-LD with `@type: Dataset`.
4. Writes both files next to the notebook.

## 1. Inspect the dataset

In [17]:
from pathlib import Path
import hashlib
import json
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
DATASET_PATH = NOTEBOOK_DIR / "Dataset" / "OnlyTrivial_dt (1).csv"

assert DATASET_PATH.exists(), f"Dataset not found at {DATASET_PATH}"

df = pd.read_csv(DATASET_PATH, low_memory=False)
size_bytes = DATASET_PATH.stat().st_size

sha256 = hashlib.sha256()
with DATASET_PATH.open("rb") as fh:
    for chunk in iter(lambda: fh.read(1 << 20), b""):
        sha256.update(chunk)
checksum = sha256.hexdigest()

summary = {
    "rows": int(len(df)),
    "columns": int(df.shape[1]),
    "size_bytes": size_bytes,
    "size_mb": round(size_bytes / (1024 * 1024), 2),
    "sha256": checksum,
    "refactoring_label_distribution": df["refactoring"].value_counts().to_dict(),
}
print(json.dumps(summary, indent=2))

{
  "rows": 232468,
  "columns": 53,
  "size_bytes": 51158315,
  "size_mb": 48.79,
  "sha256": "7e9584a9cdb6a29b148e1f16ac769b61864343d5abcc5688f96d297ae12e6f55",
  "refactoring_label_distribution": {
    "1": 119869,
    "0": 112599
  }
}


In [18]:
print("Columns (" + str(df.shape[1]) + "):")
for col in df.columns:
    print(f"  - {col}: {df[col].dtype}")

Columns (53):
  - refactoring: int64
  - file: str
  - class: str
  - type: str
  - cbo: int64
  - cboModified: int64
  - fanin: int64
  - fanout: int64
  - wmc: int64
  - dit: int64
  - noc: int64
  - rfc: int64
  - lcom: int64
  - lcom*: float64
  - tcc: float64
  - lcc: float64
  - totalMethodsQty: int64
  - staticMethodsQty: int64
  - publicMethodsQty: int64
  - privateMethodsQty: int64
  - protectedMethodsQty: int64
  - defaultMethodsQty: int64
  - visibleMethodsQty: int64
  - abstractMethodsQty: int64
  - finalMethodsQty: int64
  - synchronizedMethodsQty: int64
  - totalFieldsQty: int64
  - staticFieldsQty: int64
  - publicFieldsQty: int64
  - privateFieldsQty: int64
  - protectedFieldsQty: int64
  - defaultFieldsQty: int64
  - finalFieldsQty: int64
  - synchronizedFieldsQty: int64
  - nosi: int64
  - loc: int64
  - returnQty: int64
  - loopQty: int64
  - comparisonsQty: int64
  - tryCatchQty: int64
  - parenthesizedExpsQty: int64
  - stringLiteralsQty: int64
  - numbersQty: int6

## 2. Shared descriptive facts

Central place for the human-authored metadata. The XML and JSON-LD builders below consume this dictionary so the two files stay consistent.

In [19]:
META = {
    "identifier_doi": "10.5281/zenodo.6800385",
    "landing_page": "https://doi.org/10.5281/zenodo.6800385",
    "title": "Dataset of code metrics",
    "creators": [
        {"name": "DATASET1", "orcid": None, "affiliation": "UFC"},
    ],
    "publisher": "Zenodo",
    "publication_year": "2022",
    "resource_type": "Dataset",
    "description": (
        "Tabular dataset of CK software metrics computed on Java class versions. "
        "Each row captures metrics (CBO, WMC, DIT, NOC, RFC, LCOM, LCOM*, TCC, "
        "LCC, method/field counts, LOC, and AST-derived counters) for a Java class, "
        "plus a binary `refactoring` label marking whether a trivial class-level "
        "refactoring was applied at that commit. The dataset supports supervised "
        "learning of refactoring recommenders."
    ),
    "subjects": [
        "software engineering",
        "software refactoring",
        "software metrics",
        "CK metrics",
        "machine learning",
        "Java",
    ],
    "keywords": [
        "refactoring",
        "CK metrics",
        "Java",
        "supervised learning",
        "code quality",
    ],
    "language": "en",
    "license_name": "Creative Commons Attribution 4.0 International",
    "license_spdx": "CC-BY-4.0",
    "license_url": "https://creativecommons.org/licenses/by/4.0/",
    "format": "text/csv",
    "related_publication_doi": None,
    "related_publication_citation": None,
    "funder": None,
    "version": "1.0",
    "rows": summary["rows"],
    "columns": summary["columns"],
    "size_bytes": summary["size_bytes"],
    "sha256": summary["sha256"],
}
print(json.dumps({k: v for k, v in META.items() if k != "creators"}, indent=2))

{
  "identifier_doi": "10.5281/zenodo.3547639",
  "landing_page": "https://doi.org/10.5281/zenodo.3547639",
  "title": "OnlyTrivial_dt \u2014 Class-level trivial refactoring instances with CK metrics (ML4Refactoring subset)",
  "publisher": "Zenodo",
  "publication_year": "2019",
  "resource_type": "Dataset",
  "description": "Row-per-Java-class snapshot drawn from the ML4Refactoring corpus of ~11k open-source Java projects. Each row captures CK software metrics (CBO, WMC, DIT, NOC, RFC, LCOM, LCOM*, TCC, LCC, method/field counts, LOC, and AST-derived counters) for a class version, plus a binary `refactoring` label marking whether a trivial class-level refactoring was applied at that commit. The subset filters the full ML4Refactoring release to only trivial class-level refactorings (rename, extract, move class, etc.) for supervised learning of refactoring recommenders.",
  "subjects": [
    "software engineering",
    "software refactoring",
    "software metrics",
    "CK metrics",
  

## 3. Build DataCite 4.5 XML

Follows the DataCite Metadata Schema 4.5 (`http://datacite.org/schema/kernel-4`). Mandatory properties: Identifier, Creator, Title, Publisher, PublicationYear, ResourceType.

In [20]:
from lxml import etree

NS = "http://datacite.org/schema/kernel-4"
XSI = "http://www.w3.org/2001/XMLSchema-instance"
SCHEMA_LOC = (
    "http://datacite.org/schema/kernel-4 "
    "http://schema.datacite.org/meta/kernel-4.5/metadata.xsd"
)

nsmap = {None: NS, "xsi": XSI}
resource = etree.Element(f"{{{NS}}}resource", nsmap=nsmap)
resource.set(f"{{{XSI}}}schemaLocation", SCHEMA_LOC)

identifier = etree.SubElement(resource, "identifier", identifierType="DOI")
identifier.text = META["identifier_doi"]

creators_el = etree.SubElement(resource, "creators")
for c in META["creators"]:
    creator = etree.SubElement(creators_el, "creator")
    name = etree.SubElement(creator, "creatorName")
    name.text = c["name"]
    if ", " in c["name"]:
        family, _, given = c["name"].partition(", ")
        name.set("nameType", "Personal")
        etree.SubElement(creator, "familyName").text = family
        etree.SubElement(creator, "givenName").text = given
    if c.get("orcid"):
        nid = etree.SubElement(
            creator, "nameIdentifier",
            nameIdentifierScheme="ORCID",
            schemeURI="https://orcid.org/",
        )
        nid.text = f"https://orcid.org/{c['orcid']}"
    if c.get("affiliation"):
        etree.SubElement(creator, "affiliation").text = c["affiliation"]

titles = etree.SubElement(resource, "titles")
etree.SubElement(titles, "title", attrib={"{http://www.w3.org/XML/1998/namespace}lang": META["language"]}).text = META["title"]

etree.SubElement(resource, "publisher").text = META["publisher"]
etree.SubElement(resource, "publicationYear").text = META["publication_year"]

subjects_el = etree.SubElement(resource, "subjects")
for s in META["subjects"]:
    etree.SubElement(subjects_el, "subject").text = s

etree.SubElement(
    resource, "language",
).text = META["language"]

resource_type = etree.SubElement(
    resource, "resourceType", resourceTypeGeneral="Dataset",
)
resource_type.text = "Tabular dataset (CSV)"

sizes = etree.SubElement(resource, "sizes")
etree.SubElement(sizes, "size").text = f"{META['size_bytes']} bytes"
etree.SubElement(sizes, "size").text = f"{META['rows']} rows"
etree.SubElement(sizes, "size").text = f"{META['columns']} columns"

formats = etree.SubElement(resource, "formats")
etree.SubElement(formats, "format").text = META["format"]

etree.SubElement(resource, "version").text = META["version"]

rights_list = etree.SubElement(resource, "rightsList")
rights = etree.SubElement(
    rights_list, "rights",
    rightsURI=META["license_url"],
    rightsIdentifier=META["license_spdx"],
    rightsIdentifierScheme="SPDX",
)
rights.text = META["license_name"]

descriptions = etree.SubElement(resource, "descriptions")
etree.SubElement(
    descriptions, "description", descriptionType="Abstract",
).text = META["description"]

if META.get("related_publication_doi"):
    related = etree.SubElement(resource, "relatedIdentifiers")
    etree.SubElement(
        related, "relatedIdentifier",
        relatedIdentifierType="DOI",
        relationType="IsDocumentedBy",
    ).text = META["related_publication_doi"]

xml_bytes = etree.tostring(
    resource, pretty_print=True, xml_declaration=True, encoding="UTF-8",
)
print(xml_bytes.decode("utf-8"))

<?xml version='1.0' encoding='UTF-8'?>
<resource xmlns="http://datacite.org/schema/kernel-4" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.5/metadata.xsd">
  <identifier identifierType="DOI">10.5281/zenodo.3547639</identifier>
  <creators>
    <creator>
      <creatorName nameType="Personal">Aniche, Maurício</creatorName>
      <familyName>Aniche</familyName>
      <givenName>Maurício</givenName>
      <nameIdentifier nameIdentifierScheme="ORCID" schemeURI="https://orcid.org/">https://orcid.org/0000-0003-3576-3707</nameIdentifier>
      <affiliation>Delft University of Technology</affiliation>
    </creator>
    <creator>
      <creatorName nameType="Personal">Maziero, Erick Galante</creatorName>
      <familyName>Maziero</familyName>
      <givenName>Erick Galante</givenName>
      <affiliation>Universidade Federal de Lavras</affiliation>
    </creator>
    <creator>
      <creator

In [21]:
xml_path = NOTEBOOK_DIR / "onlytrivial_datacite.xml"
xml_path.write_bytes(xml_bytes)
print(f"Wrote {xml_path} ({xml_path.stat().st_size} bytes)")

Wrote e:\Master's Notes\2nd semester\Data Management\Data Management Assignments\Data-Management\Week_1_Exercise_1_Individual_Metadata_Creation\onlytrivial_datacite.xml (3375 bytes)


## 4. Build schema.org JSON-LD

Uses `@type: Dataset` per https://schema.org/Dataset. Includes `distribution` (DataDownload) and `variableMeasured` entries derived from the CSV columns, which makes the metadata richer than the DataCite XML while staying consistent with it.

In [22]:
variable_measured = [{"@type": "PropertyValue", "name": col} for col in df.columns]

schema_org = {
    "@context": "https://schema.org/",
    "@type": "Dataset",
    "@id": META["landing_page"],
    "identifier": [
        {
            "@type": "PropertyValue",
            "propertyID": "DOI",
            "value": META["identifier_doi"],
            "url": META["landing_page"],
        }
    ],
    "name": META["title"],
    "description": META["description"],
    "url": META["landing_page"],
    "version": META["version"],
    "datePublished": META["publication_year"],
    "inLanguage": META["language"],
    "license": META["license_url"],
    "keywords": META["keywords"],
    "creator": [
        {
            "@type": "Person",
            "name": c["name"],
            **({"identifier": f"https://orcid.org/{c['orcid']}"} if c.get("orcid") else {}),
            **({"affiliation": {"@type": "Organization", "name": c["affiliation"]}} if c.get("affiliation") else {}),
        }
        for c in META["creators"]
    ],
    "publisher": {
        "@type": "Organization",
        "name": META["publisher"],
        "url": "https://zenodo.org/",
    },
    "isAccessibleForFree": True,
    "encodingFormat": META["format"],
    "distribution": [
        {
            "@type": "DataDownload",
            "encodingFormat": META["format"],
            "contentUrl": META["landing_page"],
            "contentSize": f"{META['size_bytes']} B",
            "sha256": META["sha256"],
        }
    ],
    "variableMeasured": variable_measured,
    "size": {
        "@type": "QuantitativeValue",
        "unitText": "rows",
        "value": META["rows"],
    },
}

if META.get("related_publication_citation"):
    schema_org["citation"] = META["related_publication_citation"]
if META.get("related_publication_doi"):
    schema_org["isBasedOn"] = f"https://doi.org/{META['related_publication_doi']}"

print(json.dumps(schema_org, indent=2)[:2000] + "\n...")

{
  "@context": "https://schema.org/",
  "@type": "Dataset",
  "@id": "https://doi.org/10.5281/zenodo.3547639",
  "identifier": [
    {
      "@type": "PropertyValue",
      "propertyID": "DOI",
      "value": "10.5281/zenodo.3547639",
      "url": "https://doi.org/10.5281/zenodo.3547639"
    }
  ],
  "name": "OnlyTrivial_dt \u2014 Class-level trivial refactoring instances with CK metrics (ML4Refactoring subset)",
  "description": "Row-per-Java-class snapshot drawn from the ML4Refactoring corpus of ~11k open-source Java projects. Each row captures CK software metrics (CBO, WMC, DIT, NOC, RFC, LCOM, LCOM*, TCC, LCC, method/field counts, LOC, and AST-derived counters) for a class version, plus a binary `refactoring` label marking whether a trivial class-level refactoring was applied at that commit. The subset filters the full ML4Refactoring release to only trivial class-level refactorings (rename, extract, move class, etc.) for supervised learning of refactoring recommenders.",
  "url": 

In [23]:
jsonld_path = NOTEBOOK_DIR / "onlytrivial_schemaorg.json"
jsonld_path.write_text(json.dumps(schema_org, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote {jsonld_path} ({jsonld_path.stat().st_size} bytes)")

Wrote e:\Master's Notes\2nd semester\Data Management\Data Management Assignments\Data-Management\Week_1_Exercise_1_Individual_Metadata_Creation\onlytrivial_schemaorg.json (7197 bytes)


## 5. Verify the outputs

In [24]:
parsed_xml = etree.fromstring(xml_path.read_bytes())
assert parsed_xml.tag.endswith("resource"), "Root element must be <resource>"

parsed_jsonld = json.loads(jsonld_path.read_text(encoding="utf-8"))
assert parsed_jsonld["@type"] == "Dataset"
assert parsed_jsonld["@context"].startswith("https://schema.org")

print("DataCite XML — well-formed ✅")
print("schema.org JSON-LD — parses and declares @type: Dataset ✅")

DataCite XML — well-formed ✅
schema.org JSON-LD — parses and declares @type: Dataset ✅
